# 03 — Production-Quality Preprocessing & Train/Test Strategy

## 1. Objective
The goal of this notebook is to establish a **leakage-safe production preprocessing pipeline** and a **stratified train/test splitting strategy**.

Key deliverables:
1. Load dataset with in-memory data quality handling (`trestbps == 0` -> NaN, `chol == 0` -> NaN).
2. Perform stratified train/test split ($80\%$ train / $20\%$ test, $N_{\text{train}}=734$, $N_{\text{test}}=184$).
3. Construct scikit-learn `ColumnTransformer` (`SimpleImputer`, `StandardScaler`, `OneHotEncoder`).
4. Fit preprocessing transformers **strictly on training data (`X_train`)** to eliminate data leakage.
5. Transform both training and testing datasets and verify $0$ remaining NaNs.


## 2. Load Modeling Data
We load the modeling dataset using `src.data_loader.prepare_model_dataframe()`, which handles in-memory NaN conversions without altering `data/raw/heart_disease.csv`.


In [1]:
import sys
from pathlib import Path

# Locate project root containing src/
root_dir = Path.cwd()
for p in [root_dir] + list(root_dir.parents):
    if (p / 'src' / 'config.py').exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

import pandas as pd
import numpy as np

from src.config import (
    DATASET_PATH,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
    BOOLEAN_FEATURES,
    MODEL_FEATURES,
    BINARY_TARGET,
    TEST_SIZE,
    RANDOM_STATE,
)
from src.data_loader import prepare_model_dataframe, load_model_data, split_data
from src.preprocessing import build_preprocessor, build_pipeline

# Load modeling DataFrame
df_model = prepare_model_dataframe()
print(f"Modeling DataFrame loaded successfully: {df_model.shape[0]} rows x {df_model.shape[1]} columns")


Modeling DataFrame loaded successfully: 918 rows x 12 columns


## 3. Data-Quality Transformations
Verify in-memory data quality handling:
- `chol == 0` converted to `NaN` (172 rows)
- `trestbps == 0` converted to `NaN` (0 rows in this dataset; min `trestbps` is 80 mm Hg)
- Negative `oldpeak` preserved (12 rows, min -2.6 mm)
- Raw CSV on disk remains unmodified.


In [2]:
print("Missing Values in Modeling DataFrame:")
print(df_model[['trestbps', 'chol', 'oldpeak']].isna().sum())
print("\nNegative oldpeak count:", (df_model['oldpeak'] < 0).sum())


Missing Values in Modeling DataFrame:
trestbps      0
chol        172
oldpeak       0
dtype: int64

Negative oldpeak count: 12


## 4. Define X and y
Separate feature matrix `X` (10 model features) and target vector `y` (`target`).
Verify `num` and `target` are excluded from `X`.


In [3]:
X, y = load_model_data()
print("Feature matrix X columns:", list(X.columns))
print("'num' in X?", 'num' in X.columns)
print("'target' in X?", 'target' in X.columns)


Feature matrix X columns: ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'sex', 'cp', 'restecg', 'fbs', 'exang']
'num' in X? False
'target' in X? False


## 5. Inspect X/y
Inspect shape and missing value counts in feature matrix `X`.


In [4]:
print(f"X shape: {X.shape}, y shape: {y.shape}")
print("\nMissing Values per Feature in X:")
print(X.isna().sum())


X shape: (918, 10), y shape: (918,)

Missing Values per Feature in X:
age           0
trestbps      0
chol        172
thalch        0
oldpeak       0
sex           0
cp            0
restecg       0
fbs           0
exang         0
dtype: int64


## 6. Train/Test Split
Split `X` and `y` into $80\%$ training set ($734$ samples) and $20\%$ testing set ($184$ samples) using `split_data()`.


In [5]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=True)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape},  y_test shape:  {y_test.shape}")


X_train shape: (734, 10), y_train shape: (734,)
X_test shape:  (184, 10),  y_test shape:  (184,)


## 7. Verify Class Distribution in Train/Test
Verify that stratified splitting maintains the exact positive class ratio ($pprox 55.34\%$) across both train and test splits.


In [6]:
train_dist = pd.DataFrame({'Count': y_train.value_counts(), 'Percentage (%)': (y_train.value_counts(normalize=True)*100).round(2)})
test_dist = pd.DataFrame({'Count': y_test.value_counts(), 'Percentage (%)': (y_test.value_counts(normalize=True)*100).round(2)})

print("--- Train Set Target Distribution ---")
print(train_dist)
print("\n--- Test Set Target Distribution ---")
print(test_dist)


--- Train Set Target Distribution ---
        Count  Percentage (%)
target                       
1         406           55.31
0         328           44.69

--- Test Set Target Distribution ---
        Count  Percentage (%)
target                       
1         102           55.43
0          82           44.57


## 8. Build Preprocessing Transformer
Construct scikit-learn `ColumnTransformer` with:
- `num`: `SimpleImputer(strategy='median', add_indicator=True)` $ightarrow$ `StandardScaler()`
- `cat`: `SimpleImputer(strategy='most_frequent')` $ightarrow$ `OneHotEncoder(handle_unknown='ignore')`
- `bool`: `FunctionTransformer(_cast_boolean_to_int)` $ightarrow$ `SimpleImputer(strategy='most_frequent')`


In [7]:
preprocessor = build_preprocessor()
print("Preprocessor constructed successfully:")
print(preprocessor)


Preprocessor constructed successfully:
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'trestbps', 'chol', 'thalch',
                                  'oldpeak']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['sex', 'cp', 'restecg']),


## 9. Fit Transformer ONLY on Training Data
> **CRITICAL LEAKAGE PREVENTION:** The preprocessor is fit **strictly on `X_train`**. Imputation medians and scaling parameters are calculated exclusively from training samples.


In [8]:
preprocessor.fit(X_train)
print("Preprocessor fit completed on X_train.")


Preprocessor fit completed on X_train.


## 10. Transform Training Data
Transform `X_train` into preprocessed NumPy array `X_train_transformed`.


In [9]:
X_train_transformed = preprocessor.transform(X_train)
print(f"X_train_transformed shape: {X_train_transformed.shape}")


X_train_transformed shape: (734, 16)


## 11. Transform Test Data
Transform `X_test` into `X_test_transformed` using the **pre-fitted** preprocessor.


In [10]:
X_test_transformed = preprocessor.transform(X_test)
print(f"X_test_transformed shape: {X_test_transformed.shape}")


X_test_transformed shape: (184, 16)


## 12. Inspect Transformed Feature Dimensions
Verify dimensions:
- Training split: $734$ rows $\times 16$ features
- Testing split: $184$ rows $\times 16$ features


In [11]:
print(f"Train Transformed Array Shape: {X_train_transformed.shape}")
print(f"Test Transformed Array Shape:  {X_test_transformed.shape}")


Train Transformed Array Shape: (734, 16)
Test Transformed Array Shape:  (184, 16)


## 13. Display Transformed Feature Names
Inspect feature names generated by the `ColumnTransformer`.


In [12]:
feature_names = preprocessor.get_feature_names_out()
for idx, name in enumerate(feature_names, 1):
    print(f"  {idx:2d}. {name}")


   1. num__age
   2. num__trestbps
   3. num__chol
   4. num__thalch
   5. num__oldpeak
   6. num__missingindicator_chol
   7. cat__sex_Female
   8. cat__sex_Male
   9. cat__cp_asymptomatic
  10. cat__cp_non-anginal
  11. cat__cp_typical angina
  12. cat__restecg_lv hypertrophy
  13. cat__restecg_normal
  14. cat__restecg_st-t abnormality
  15. bool__fbs
  16. bool__exang


## 14. Verify No NaNs Remain After Preprocessing
Confirm zero missing values in transformed training and testing matrices.


In [13]:
nan_train = np.isnan(X_train_transformed).sum()
nan_test = np.isnan(X_test_transformed).sum()
print(f"NaN count in X_train_transformed: {nan_train}")
print(f"NaN count in X_test_transformed:  {nan_test}")
assert nan_train == 0 and nan_test == 0, "Preprocessing error: NaNs remain!"


NaN count in X_train_transformed: 0
NaN count in X_test_transformed:  0


## 15. Explain Leakage Prevention

### How Data Leakage is Prevented
1. **Separation Before Transformation:** The train/test split is performed **before** any fitting, imputation, or scaling occurs.
2. **Independent Test Pipeline:** Test data (`X_test`) is never used to compute feature medians, means, or standard deviations.
3. **Pipeline Encapsulation:** In production modeling, the preprocessor will be encapsulated inside an `sklearn.pipeline.Pipeline`, guaranteeing that cross-validation folds fit preprocessors strictly on training folds.


## 16. Explain Every Preprocessing Decision

- **Numerical Imputation (`SimpleImputer(strategy='median', add_indicator=True)`):** Robust to skewed distributions. `add_indicator=True` generates `missingindicator_chol` to retain missingness information as an explicit predictive signal.
- **Numerical Scaling (`StandardScaler`):** Standardizes numerical features to zero mean and unit variance.
- **Categorical Imputation & Encoding (`OneHotEncoder(handle_unknown='ignore')`):** Imputes most frequent category and one-hot encodes categorical values (`sex`, `cp`, `restecg`). `handle_unknown='ignore'` prevents runtime crashes on unseen categories during deployment.
- **Boolean Pipeline:** Casts boolean flags (`fbs`, `exang`) to numeric 0/1 integers.


## 17. Summary
In Phase 3:
1. Implemented in-memory data quality handling (`trestbps == 0` -> NaN, `chol == 0` -> NaN).
2. Executed stratified 80/20 train/test split ($734$ train, $184$ test).
3. Constructed leakage-safe `ColumnTransformer` with `SimpleImputer`, `StandardScaler`, and `OneHotEncoder`.
4. Verified zero NaNs remaining post-transformation across both splits.
